# 📚 **Sinhala Buddhist Corpus Builder v2.0**
## Advanced PDF Text Extraction with Intelligent Page Filtering

### **Key Features:**
- ✅ **Smart Unicode vs Vision OCR Detection**
- ✅ **Adaptive Page Filtering** (Post-OCR content validation)
- ✅ **Organized Folder Structure** (Per-PDF extraction tracking)
- ✅ **Multi-Criteria Content Detection**
- ✅ **Cost Optimization** (Skip non-content pages)

---

## 🔧 **1. Setup & Installation**

In [ ]:
# ========================================
# INSTALL DEPENDENCIES
# ========================================

!pip install -q PyMuPDF pillow google-cloud-vision tqdm

print("✅ All dependencies installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.1/529.1 kB 27.4 MB/s eta 0:00:00
✅ All dependencies installed successfully!


## 📂 **2. Configuration & Folder Structure**

In [ ]:
import os
import re
import json
import fitz  # PyMuPDF
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from tqdm.auto import tqdm
import gc

# ========================================
# FOLDER STRUCTURE CONFIGURATION
# ========================================

BASE_DIR = Path("/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/")  # Change this to your desired location
PDF_FOLDER = BASE_DIR / "data" / "raw" / "tripitaka"

# Organized extraction folders /sinhala_corpus
EXTRACTION_BASE = BASE_DIR / "data" / "extractions"
RAW_TEXT_DIR = EXTRACTION_BASE / "1_raw_text"
CLEANED_TEXT_DIR = EXTRACTION_BASE / "2_cleaned_text"
FINAL_CORPUS_DIR = EXTRACTION_BASE / "3_final_corpus"

# Logs and metadata
LOGS_DIR = EXTRACTION_BASE / "logs"
PROCESSED_PDFS_DIR = BASE_DIR / "data" / "processed_pdfs"

# Create all directories
for directory in [PDF_FOLDER, RAW_TEXT_DIR, CLEANED_TEXT_DIR, FINAL_CORPUS_DIR, LOGS_DIR, PROCESSED_PDFS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ========================================
# PROCESSING PARAMETERS
# ========================================

# Vision API settings
VISION_API_COST_PER_1000 = 1.50  # USD
MAX_RAM_MB = 3000  # Max RAM before cleanup

# Content filtering parameters
MIN_SINHALA_RATIO = 0.30  # Minimum 30% Sinhala characters
ADAPTIVE_THRESHOLD_SAMPLE_SIZE = 5  # Pages to sample from middle
THRESHOLD_PERCENTILE = 0.40  # Use 40% of average middle-page length

# Page position filtering
SKIP_FIRST_N_PAGES = 3  # Likely covers/title pages
SKIP_LAST_N_PAGES = 2   # Likely back cover/publisher info

print("✅ Configuration complete!")
print(f"\n📂 Folder Structure:")
print(f"   PDFs: {PDF_FOLDER}")
print(f"   Raw texts: {RAW_TEXT_DIR}")
print(f"   Cleaned texts: {CLEANED_TEXT_DIR}")
print(f"   Final corpus: {FINAL_CORPUS_DIR}")
print(f"   Logs: {LOGS_DIR}")

✅ Configuration complete!

📂 Folder Structure:
   PDFs: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka
   Raw texts: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/extractions/1_raw_text
   Cleaned texts: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/extractions/2_cleaned_text
   Final corpus: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/extractions/3_final_corpus
   Logs: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/extractions/logs


## 🔐 **3. Google Vision API Setup**

**Upload your service account JSON key file here.**

In [ ]:
import shutil
import os
from google.colab import drive

# Step 1: Try to unmount (if mounted)
try:
    drive.flush_and_unmount()
    print("🔄 Unmounted Drive")
except:
    print("📌 Drive was not mounted")

# Step 2: Remove the directory completely
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')
    print("🗑️  Cleared /content/drive directory")

# Step 3: Now mount fresh
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully!")

Drive not mounted, so nothing to flush and unmount.
🔄 Unmounted Drive
🗑️  Cleared /content/drive directory
Mounted at /content/drive
✅ Google Drive mounted successfully!


In [ ]:
from google.colab import files
import os

VISION_API_KEY_PATH = '/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/auth/Vision OCR thematic runner.json'

import os
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = VISION_API_KEY_PATH


# Test the API
from google.cloud import vision
client = vision.ImageAnnotatorClient()
print("✅ Vision API client initialized successfully!")

✅ Vision API client initialized successfully!


## 🧠 **4. Core Text Processing Functions**

In [ ]:
# ========================================
# SINHALA TEXT UTILITIES
# ========================================

def calculate_sinhala_ratio(text):
    """Calculate the ratio of Sinhala characters in text"""
    if not text:
        return 0.0

    # Sinhala Unicode range: 0D80-0DFF
    sinhala_chars = sum(1 for char in text if '\u0D80' <= char <= '\u0DFF')
    total_chars = len(text.replace(' ', '').replace('\n', ''))

    return sinhala_chars / total_chars if total_chars > 0 else 0.0

def has_meaningful_sinhala(text, min_ratio=MIN_SINHALA_RATIO):
    """Check if text has meaningful Sinhala content"""
    return calculate_sinhala_ratio(text) >= min_ratio

# ========================================
# TEXT CLEANING
# ========================================

def clean_text(text):
    """Clean extracted text while preserving Sinhala"""
    if not text:
        return ""

    # Remove excessive whitespace
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)

    # Remove page numbers (standalone numbers)
    text = re.sub(r'^\d+$', '', text, flags=re.MULTILINE)

    # Remove common footer/header patterns
    text = re.sub(r'\n[-_=]{3,}\n', '\n', text)

    return text.strip()

print("✅ Text processing functions loaded!")

✅ Text processing functions loaded!


## 🎯 **5. Intelligent Page Filtering System**

### **Multi-Criteria Content Validation:**
1. **Adaptive Threshold**: Uses middle pages to establish content baseline
2. **Position Filtering**: Skips likely cover/back pages
3. **Pattern Detection**: Identifies TOC, copyright, publisher info
4. **Language Validation**: Ensures sufficient Sinhala content
5. **Structure Analysis**: Detects abnormal formatting

In [ ]:
# ========================================
# ADAPTIVE THRESHOLD CALCULATOR
# ========================================

def calculate_adaptive_threshold(page_texts, total_pages):
    """
    Calculate content threshold based on middle pages
    Strategy: Sample pages from the middle 50% of the document
    """
    if total_pages < 10:
        # For short documents, use a fixed threshold
        return 200  # characters

    # Define middle region (25% - 75% of document)
    start_idx = int(total_pages * 0.25)
    end_idx = int(total_pages * 0.75)

    # Sample pages from middle
    middle_pages = list(range(start_idx, end_idx))
    sample_indices = middle_pages[::max(1, len(middle_pages)//ADAPTIVE_THRESHOLD_SAMPLE_SIZE)]
    sample_indices = sample_indices[:ADAPTIVE_THRESHOLD_SAMPLE_SIZE]

    # Get lengths of sample pages
    sample_lengths = []
    for idx in sample_indices:
        if idx < len(page_texts) and page_texts[idx]:
            sample_lengths.append(len(page_texts[idx]))

    if not sample_lengths:
        return 200  # Default fallback

    # Calculate threshold as percentage of average
    avg_length = sum(sample_lengths) / len(sample_lengths)
    threshold = int(avg_length * THRESHOLD_PERCENTILE)

    # Ensure reasonable bounds
    threshold = max(100, min(threshold, 2000))

    return threshold

# ========================================
# NON-CONTENT PATTERN DETECTION
# ========================================

def contains_toc_markers(text):
    """
    Check if text contains Table of Contents markers
    """
    if not text:
        return False

    text_lower = text.lower()

    # TOC keyword indicators
    toc_keywords = [
        ('contents', 3),  # (keyword, min_occurrences)
        ('පටුන', 3),
        ('අන්තර්ගතය', 1),
        ('පරිවිඩිය', 2),
        ('පිටුව', 5),  # "page" mentioned many times
    ]

    for keyword, min_count in toc_keywords:
        if text_lower.count(keyword) >= min_count:
            return True

    # Dot leaders detection (common in TOC: "Chapter 1 ..... 5")
    if re.search(r'\.{3,}|…{2,}', text):
        dot_count = len(re.findall(r'\.{3,}|…{2,}', text))
        if dot_count >= 3:
            return True

    return False

def contains_cover_markers(text):
    """
    Check if text contains cover page markers
    """
    if not text:
        return False

    text_lower = text.lower()

    # Cover page indicators (English + Sinhala)
    cover_patterns = [
        'published by', 'publisher', 'copyright', '©', 'isbn',
        'all rights reserved', 'first edition', 'second edition',
        'ප්‍රකාශකයා', 'ප්‍රකාශන', 'මුද්‍රණ', 'සංස්කරණය', 'පිටපත',
        'ප්‍රථම මුද්‍රණය', 'ප්‍රථම සංස්කරණය'
    ]

    for pattern in cover_patterns:
        if pattern in text_lower:
            return True

    return False

def contains_end_matter_markers(text):
    """
    Check if text contains end matter markers (bibliography, index, etc.)
    """
    if not text:
        return False

    text_lower = text.lower()

    # End matter patterns
    end_matter_patterns = [
        'bibliography', 'references', 'index', 'about the author',
        'ග්‍රන්ථ නාමාවලිය', 'සටහන්', 'ලේඛක ගැන'
    ]

    for pattern in end_matter_patterns:
        # Check if pattern is at the start or text is short with this pattern
        if text_lower.startswith(pattern) or (pattern in text_lower and len(text) < 600):
            return True

    return False

def detect_non_content_patterns(text):
    """
    Detect patterns indicating non-content pages
    Returns: (is_non_content, reason)
    """
    if not text or len(text.strip()) < 20:
        return True, "too_short"

    # Check for cover patterns
    if contains_cover_markers(text):
        return True, "cover_pattern"

    # Check for TOC patterns
    if contains_toc_markers(text):
        return True, "toc_pattern"

    # Check for end matter
    if contains_end_matter_markers(text):
        return True, "end_matter"

    # Excessive page numbers (standalone numbers on their own lines)
    standalone_numbers = re.findall(r'^\s*\d+\s*$', text, re.MULTILINE)
    if len(standalone_numbers) > 5:
        return True, "excessive_page_numbers"

    # Dedication/Preface indicators (only if short)
    if len(text) < 800:
        dedication_patterns = [
            'dedicated to', 'dedication', 'preface', 'foreword',
            'acknowledgment', 'acknowledgement',
            'කෘතඥතාව', 'පෙරවදන', 'කැපවීම'
        ]
        text_lower = text.lower()
        for pattern in dedication_patterns:
            if pattern in text_lower:
                return True, f"front_matter:{pattern}"

    return False, "valid_content"

# ========================================
# HYBRID MULTI-SIGNAL PAGE VALIDATOR
# ========================================

def is_valid_content_page(text, page_num, total_pages, threshold):
    """
    Hybrid validation combining adaptive threshold with pattern detection

    Uses multiple signals:
    1. Adaptive threshold (primary filter)
    2. Sinhala character density (language validation)
    3. Position heuristics (structural)
    4. Pattern detection (semantic)
    5. Text structure analysis (stylistic)

    Returns: (is_valid, reason)
    """

    # Signal 0: Basic sanity check
    if not text or len(text.strip()) < 50:
        return False, "too_short"

    # Signal 1: Position-based filtering (soft filter, not absolute)
    position_penalty = 1.0
    if page_num < SKIP_FIRST_N_PAGES:
        position_penalty = 0.5  # Penalize but don't auto-reject
    elif page_num >= total_pages - SKIP_LAST_N_PAGES:
        position_penalty = 0.5

    # Signal 2: Explicit non-content pattern detection (hard filter)
    is_non_content, pattern_reason = detect_non_content_patterns(text)
    if is_non_content:
        return False, pattern_reason

    # Signal 3: Sinhala language validation
    sinhala_ratio = calculate_sinhala_ratio(text)
    if sinhala_ratio < MIN_SINHALA_RATIO:
        # Too little Sinhala content
        return False, f"low_sinhala:{sinhala_ratio:.2f}"

    # Signal 4: Adaptive threshold check with exceptions
    text_length = len(text)
    length_ratio = text_length / threshold if threshold > 0 else 0

    # Apply position penalty to effective length
    effective_length_ratio = length_ratio * position_penalty

    # Decision logic
    if effective_length_ratio >= THRESHOLD_PERCENTILE:
        # Text is long enough (considering position penalty)
        return True, "valid"

    elif effective_length_ratio >= 0.3 and sinhala_ratio >= 0.7:
        # Exception: High Sinhala density (likely verse pages)
        # Even if shorter, keep it if it's predominantly Sinhala
        return True, "valid_high_sinhala"

    elif effective_length_ratio >= 0.2 and sinhala_ratio >= 0.8:
        # Exception: Very high Sinhala density (short suttas, verses)
        return True, "valid_verse"

    else:
        # Below threshold and no exceptions apply
        return False, f"below_threshold:{text_length}<{int(threshold*THRESHOLD_PERCENTILE)}"

print("✅ Hybrid multi-signal page filtering system loaded!")
print("   • Adaptive threshold (primary filter)")
print("   • Pattern detection (TOC, cover, end matter)")
print("   • Sinhala density validation")
print("   • Position-based penalties")
print("   • High-Sinhala exceptions (verse pages)")

✅ Hybrid multi-signal page filtering system loaded!
   • Adaptive threshold (primary filter)
   • Pattern detection (TOC, cover, end matter)
   • Sinhala density validation
   • Position-based penalties
   • High-Sinhala exceptions (verse pages)


## 📄 **6. PDF Text Extraction Functions**

In [ ]:
# ========================================
# UNICODE EXTRACTION (FREE)
# ========================================

def extract_text_unicode(pdf_path, page_num):
    """Extract text using PyMuPDF (free, fast)"""
    try:
        doc = fitz.open(pdf_path)
        page = doc[page_num]
        text = page.get_text("text")
        doc.close()
        return text.strip()
    except Exception as e:
        return None

def check_unicode_availability(pdf_path, sample_pages=3):
    """
    Check if PDF has extractable Unicode text
    Sample first few pages to determine
    """
    try:
        doc = fitz.open(pdf_path)
        total_pages = len(doc)

        # Sample pages from beginning and middle
        check_pages = [0, min(5, total_pages//2), min(10, total_pages-1)]
        check_pages = [p for p in check_pages if p < total_pages]

        total_text_length = 0
        for page_num in check_pages:
            text = extract_text_unicode(pdf_path, page_num)
            if text:
                total_text_length += len(text)

        doc.close()

        # If we got substantial text, Unicode is available
        avg_length = total_text_length / len(check_pages)
        return avg_length > 100  # At least 100 chars per page on average

    except Exception as e:
        return False

# ========================================
# VISION OCR (PAID, FOR IMAGE-BASED PDFs)
# ========================================

def extract_text_vision_ocr(pdf_path, page_num):
    """
    Extract text using Google Vision API OCR
    Use for scanned/image-based PDFs
    """
    try:
        from google.cloud import vision
        from PIL import Image
        import io

        # Convert PDF page to image
        doc = fitz.open(pdf_path)
        page = doc[page_num]
        pix = page.get_pixmap(dpi=300)

        # Convert to PIL Image
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        # Convert to bytes for Vision API
        img_byte_arr = io.BytesIO()
        img.save(img_byte_arr, format='PNG')
        img_bytes = img_byte_arr.getvalue()

        doc.close()

        # Call Vision API
        client = vision.ImageAnnotatorClient()
        image = vision.Image(content=img_bytes)
        response = client.document_text_detection(image=image)

        if response.error.message:
            raise Exception(response.error.message)

        text = response.full_text_annotation.text
        return text.strip()

    except Exception as e:
        print(f"    ⚠️  Vision OCR failed for page {page_num}: {e}")
        return None

print("✅ Extraction functions loaded!")

✅ Extraction functions loaded!


## 🚀 **7. Main Processing Pipeline**

In [ ]:
# ========================================
# PROCESS SINGLE PDF WITH FILTERING
# ========================================

def process_pdf_with_filtering(pdf_path, use_vision_ocr=False):
    """
    Process PDF with intelligent page filtering
    """
    pdf_name = pdf_path.stem
    print(f"\n{'='*70}")
    print(f"📖 Processing: {pdf_name}")
    print(f"{'='*70}")

    # Create PDF-specific folders
    pdf_raw_folder = RAW_TEXT_DIR / pdf_name
    pdf_cleaned_folder = CLEANED_TEXT_DIR / pdf_name
    pdf_raw_folder.mkdir(exist_ok=True)
    pdf_cleaned_folder.mkdir(exist_ok=True)

    # Processing stats
    stats = {
        'pdf_name': pdf_name,
        'total_pages': 0,
        'extracted_pages': 0,
        'filtered_pages': 0,
        'filter_reasons': defaultdict(int),
        'extraction_method': 'unicode' if not use_vision_ocr else 'vision_ocr',
        'vision_api_calls': 0,
        'vision_api_cost': 0.0,
        'adaptive_threshold': 0
    }

    try:
        doc = fitz.open(pdf_path)
        stats['total_pages'] = len(doc)
        doc.close()

        print(f"📄 Total pages: {stats['total_pages']}")
        print(f"🔧 Extraction method: {stats['extraction_method'].upper()}")

        # PHASE 1: Extract all pages (raw)
        print(f"\n🔄 Phase 1: Extracting all pages...")
        all_page_texts = []

        for page_num in tqdm(range(stats['total_pages']), desc="Extracting"):
            if use_vision_ocr:
                text = extract_text_vision_ocr(pdf_path, page_num)
                stats['vision_api_calls'] += 1
            else:
                text = extract_text_unicode(pdf_path, page_num)

            all_page_texts.append(text if text else "")

            # Save raw extraction
            raw_file = pdf_raw_folder / f"{page_num + 1}.txt"
            with open(raw_file, 'w', encoding='utf-8') as f:
                f.write(text if text else "")

        # Calculate Vision API cost
        if use_vision_ocr:
            stats['vision_api_cost'] = (stats['vision_api_calls'] / 1000) * VISION_API_COST_PER_1000

        # PHASE 2: Calculate adaptive threshold
        print(f"\n🎯 Phase 2: Calculating content threshold...")
        threshold = calculate_adaptive_threshold(all_page_texts, stats['total_pages'])
        stats['adaptive_threshold'] = threshold
        print(f"   ✓ Adaptive threshold: {threshold} characters")

        # PHASE 3: Filter and validate pages
        print(f"\n🔍 Phase 3: Filtering non-content pages...")
        valid_pages = []

        for page_num, text in enumerate(all_page_texts):
            is_valid, reason = is_valid_content_page(
                text, page_num, stats['total_pages'], threshold
            )

            if is_valid:
                valid_pages.append((page_num, text))
                stats['extracted_pages'] += 1

                # Save cleaned text
                cleaned_text = clean_text(text)
                cleaned_file = pdf_cleaned_folder / f"{page_num + 1}.txt"
                with open(cleaned_file, 'w', encoding='utf-8') as f:
                    f.write(cleaned_text)
            else:
                stats['filtered_pages'] += 1
                stats['filter_reasons'][reason] += 1

        # Results
        print(f"\n✅ Extraction complete!")
        print(f"   📊 Valid content pages: {stats['extracted_pages']}/{stats['total_pages']}")
        print(f"   🗑️  Filtered pages: {stats['filtered_pages']}")

        if stats['filter_reasons']:
            print(f"\n   Filter breakdown:")
            for reason, count in sorted(stats['filter_reasons'].items(), key=lambda x: -x[1])[:5]:
                print(f"      • {reason}: {count} pages")

        if use_vision_ocr:
            print(f"\n   💰 Vision API cost: ${stats['vision_api_cost']:.4f}")

        # Combine valid pages
        combined_text = "\n\n".join([text for _, text in valid_pages])

        return combined_text, stats

    except Exception as e:
        print(f"\n❌ Error processing {pdf_name}: {e}")
        return None, stats

print("✅ Main processing pipeline loaded!")

✅ Main processing pipeline loaded!


## 🎬 **8. Run Corpus Builder**

**Upload your PDFs to the `pdfs` folder, then run this cell.**

In [ ]:
# ========================================
# MAIN EXECUTION
# ========================================

print("\n" + "="*80)
print("🚀 STARTING SINHALA BUDDHIST CORPUS BUILDER")
print("="*80)

# Get all PDFs
pdf_files = sorted(list(PDF_FOLDER.rglob("*.pdf")))
print(f"\n📚 Found {len(pdf_files)} PDF files")

if len(pdf_files) == 0:
    print("\n⚠️  No PDFs found! Please upload PDFs to:")
    print(f"   {PDF_FOLDER}")
else:
    # Initialize tracking
    all_texts = []
    processing_log = []
    total_cost = 0.0

    # Process each PDF
    for pdf_path in pdf_files:
        # Check if Unicode extraction works
        has_unicode = check_unicode_availability(pdf_path)
        use_vision = not has_unicode

        if use_vision:
            print(f"\n⚠️  No Unicode text detected - will use Vision OCR")

        # Process the PDF
        text, stats = process_pdf_with_filtering(pdf_path, use_vision_ocr=use_vision)

        if text:
            all_texts.append(text)
            total_cost += stats['vision_api_cost']
            processing_log.append(stats)

            # Move processed PDF
            try:
                pdf_path.rename(PROCESSED_PDFS_DIR / pdf_path.name)
            except Exception as e:
                print(f"   ⚠️  Could not move PDF: {e}")

        # Memory cleanup
        gc.collect()

    # ========================================
    # CREATE FINAL CORPUS
    # ========================================

    if all_texts:
        print(f"\n\n{'='*80}")
        print("📝 CREATING FINAL CORPUS")
        print("="*80)

        # Create timestamped folder
        now = datetime.now()
        corpus_folder = FINAL_CORPUS_DIR / str(now.year) / f"{now.month:02d}" / f"{now.day:02d}"
        corpus_folder.mkdir(parents=True, exist_ok=True)

        # Combine all texts
        combined_corpus = "\n\n==========\n\n".join(all_texts)

        # Save corpus
        corpus_file = corpus_folder / "sinhala_buddhist_corpus.txt"
        with open(corpus_file, 'w', encoding='utf-8') as f:
            f.write(combined_corpus)

        print(f"\n✅ Final corpus saved to: {corpus_file}")

        # Save processing log
        log_file = LOGS_DIR / f"processing_log_{now.strftime('%Y%m%d_%H%M%S')}.json"
        with open(log_file, 'w', encoding='utf-8') as f:
            json.dump(processing_log, f, indent=2, ensure_ascii=False)

        print(f"✅ Processing log saved to: {log_file}")

        # ========================================
        # FINAL STATISTICS
        # ========================================

        print(f"\n\n{'='*80}")
        print("📊 FINAL STATISTICS")
        print("="*80)

        total_pages = sum(s['total_pages'] for s in processing_log)
        extracted_pages = sum(s['extracted_pages'] for s in processing_log)
        filtered_pages = sum(s['filtered_pages'] for s in processing_log)

        print(f"\n📚 Books processed: {len(processing_log)}")
        print(f"📄 Total pages: {total_pages:,}")
        print(f"✅ Valid content pages: {extracted_pages:,} ({extracted_pages/total_pages*100:.1f}%)")
        print(f"🗑️  Filtered pages: {filtered_pages:,} ({filtered_pages/total_pages*100:.1f}%)")

        print(f"\n📝 Corpus statistics:")
        print(f"   Total characters: {len(combined_corpus):,}")
        print(f"   Total words: {len(combined_corpus.split()):,}")
        print(f"   Sinhala ratio: {calculate_sinhala_ratio(combined_corpus):.1%}")

        if total_cost > 0:
            print(f"\n💰 Vision API cost: ${total_cost:.4f}")

        print(f"\n✨ Corpus building complete! ✨\n")
    else:
        print("\n❌ No valid text extracted from any PDFs")


🚀 STARTING SINHALA BUDDHIST CORPUS BUILDER

📚 Found 67 PDF files

📖 Processing: බාගතකර_ගැනීම
📄 Total pages: 646
🔧 Extraction method: UNICODE

🔄 Phase 1: Extracting all pages...


Extracting:   0%|          | 0/646 [00:00<?, ?it/s]


❌ Error processing බාගතකර_ගැනීම: [Errno 5] Input/output error: '/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/extractions/1_raw_text/බාගතකර_ගැනීම/277.txt'

📖 Processing: බාගතකර_ගැනීම
📄 Total pages: 21
🔧 Extraction method: UNICODE

🔄 Phase 1: Extracting all pages...


Extracting:   0%|          | 0/21 [00:00<?, ?it/s]


🎯 Phase 2: Calculating content threshold...
   ✓ Adaptive threshold: 300 characters

🔍 Phase 3: Filtering non-content pages...

✅ Extraction complete!
   📊 Valid content pages: 19/21
   🗑️  Filtered pages: 2

   Filter breakdown:
      • low_sinhala:0.01: 1 pages
      • below_threshold:147<120: 1 pages

📖 Processing: අංගුත්තර_නිකාය_1
📄 Total pages: 661
🔧 Extraction method: UNICODE

🔄 Phase 1: Extracting all pages...


...

Extracting:   0%|          | 0/743 [00:00<?, ?it/s]


🎯 Phase 2: Calculating content threshold...
   ✓ Adaptive threshold: 100 characters

🔍 Phase 3: Filtering non-content pages...

✅ Extraction complete!
   📊 Valid content pages: 2/743
   🗑️  Filtered pages: 741

   Filter breakdown:
      • too_short: 740 pages
      • toc_pattern: 1 pages

⚠️  No Unicode text detected - will use Vision OCR

📖 Processing: බාගතකර_ගැනීම
📄 Total pages: 844
🔧 Extraction method: VISION_OCR

🔄 Phase 1: Extracting all pages...


Extracting:   0%|          | 0/844 [00:00<?, ?it/s]

    ⚠️  Vision OCR failed for page 827: Requested entity was not found.
    ⚠️  Vision OCR failed for page 829: Requested entity was not found.
    ⚠️  Vision OCR failed for page 830: Requested entity was not found.
    ⚠️  Vision OCR failed for page 832: Requested entity was not found.
    ⚠️  Vision OCR failed for page 833: Requested entity was not found.

🎯 Phase 2: Calculating content threshold...
   ✓ Adaptive threshold: 716 characters

🔍 Phase 3: Filtering non-content pages...

✅ Extraction complete!
   📊 Valid content pages: 810/844
   🗑️  Filtered pages: 34

   Filter breakdown:
      • excessive_page_numbers: 10 pages
      • too_short: 5 pages
      • cover_pattern: 3 pages
      • below_threshold:114<286: 3 pages
      • below_threshold:143<286: 2 pages

   💰 Vision API cost: $1.2660

⚠️  No Unicode text detected - will use Vision OCR

📖 Processing: බාගතකර_ගැනීම
📄 Total pages: 365
🔧 Extraction method: VISION_OCR

🔄 Phase 1: Extracting all pages...


Extracting:   0%|          | 0/365 [00:00<?, ?it/s]

## 📊 **9. Analysis & Quality Check**

In [ ]:
# ========================================
# ANALYZE FILTERING EFFECTIVENESS
# ========================================

if processing_log:
    print("\n" + "="*80)
    print("🔍 FILTERING ANALYSIS")
    print("="*80)

    # Aggregate filter reasons
    all_filter_reasons = defaultdict(int)
    for log in processing_log:
        for reason, count in log['filter_reasons'].items():
            all_filter_reasons[reason] += count

    print(f"\n📋 Top filter reasons across all PDFs:\n")
    for reason, count in sorted(all_filter_reasons.items(), key=lambda x: -x[1])[:10]:
        print(f"   {reason:30s}: {count:4d} pages")

    # Extraction method breakdown
    unicode_count = sum(1 for log in processing_log if log['extraction_method'] == 'unicode')
    vision_count = sum(1 for log in processing_log if log['extraction_method'] == 'vision_ocr')

    print(f"\n📊 Extraction method breakdown:")
    print(f"   Unicode (free): {unicode_count} PDFs")
    print(f"   Vision OCR: {vision_count} PDFs")

    if vision_count > 0:
        avg_cost = total_cost / vision_count
        print(f"   Avg cost per OCR PDF: ${avg_cost:.4f}")
else:
    print("\n⚠️  No processing data available for analysis")

## 📥 **10. Download Final Corpus**

In [ ]:
from google.colab import files
import shutil

# Create a zip of the final corpus folder
print("📦 Creating corpus archive...")

now = datetime.now()
archive_name = f"sinhala_corpus_{now.strftime('%Y%m%d')}"
shutil.make_archive(archive_name, 'zip', FINAL_CORPUS_DIR)

print(f"\n✅ Archive created: {archive_name}.zip")
print("⬇️  Downloading...")

files.download(f"{archive_name}.zip")

print("\n✨ Download complete!")

---

## 📚 **Usage Guide**

### **Workflow:**
1. Upload PDFs to `/content/sinhala_corpus/pdfs/`
2. Run Section 8 to process all PDFs
3. Check filtering statistics in Section 9
4. Download final corpus from Section 10

### **Output Structure:**
```
data/extractions/
├── 1_raw_text/
│   ├── PDF_Name_1/
│   │   ├── 1.txt
│   │   ├── 2.txt
│   │   └── ...
│   └── PDF_Name_2/
│
├── 2_cleaned_text/
│   ├── PDF_Name_1/
│   │   ├── 1.txt (only valid content pages)
│   │   └── ...
│
└── 3_final_corpus/
    └── 2025/
        └── 01/
            └── 15/
                └── sinhala_buddhist_corpus.txt
```

### **Filtering Logic:**
- **Adaptive Threshold**: Calculated from middle pages
- **Position Filtering**: Skips first 3 and last 2 pages
- **Pattern Detection**: Identifies cover, TOC, copyright pages
- **Language Validation**: Ensures 30%+ Sinhala characters
- **Quality Control**: Multiple validation layers

---

**Version:** 2.0  
**Created for:** Sinhala Buddhist Conversational Agent Project  
**License:** MIT
